# PageRank from Scratch — Eigendecomposition & Power Iteration

**Stack:** NumPy · NetworkX · Matplotlib  
**Level:** 3rd-year CSE

The web is a directed graph. PageRank answers one question: *"if you surfed the web randomly forever, how often would you land on each page?"*

That is the **stationary distribution** of a Markov chain — which is the **dominant eigenvector** of the transition matrix.

We'll:
1. Build the column-stochastic transition matrix from a real edge list
2. Handle dangling nodes (the silent bug that breaks everything)
3. Apply the damping factor — and prove *why* it guarantees convergence
4. Solve with **power iteration** and **eigendecomposition**
5. Verify both give the same result and understand why they must

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from urllib.request import urlretrieve
import gzip, os, time

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. The Math in 60 Seconds

Given a directed graph with *n* nodes:

**Step 1 — Transition matrix M** (column-stochastic):

$$M[i][j] = \frac{1}{\text{out\_degree}(j)} \quad \text{if edge } j \to i \text{ exists, else } 0$$

Each column sums to 1. "If you're at page j, go to each neighbor with equal probability."

**Step 2 — Dangling nodes** (out-degree = 0):  
Column sums to 0 — the random surfer is stuck. Fix: replace those columns with 1/n (teleport anywhere).

**Step 3 — Google Matrix** (damping factor d ≈ 0.85):

$$G = d \cdot \tilde{M} + (1-d) \cdot \frac{1}{n} \mathbf{1}\mathbf{1}^T$$

With prob d, follow a real link. With prob (1−d), jump to any random page.  
This makes every entry of G strictly positive → **Perron-Frobenius theorem** kicks in:  
unique dominant eigenvalue λ₁ = 1 with a unique positive eigenvector. That's PageRank.

**Step 4 — Solve**: PageRank = eigenvector for λ = 1, equivalently the fixed point of r = G·r.

## 2. Build the Graph

Start with a small handcrafted graph (8 nodes) — clear enough to trace by hand, enough structure to hit every edge case (dangling node, cycle, disconnected node).

In [ ]:
# Edge (u, v) = page u links TO page v
small_edges = [
    (0, 1), (0, 2),   # node 0 links out to 1 and 2
    (1, 2),           # node 1 links to 2
    (2, 0),           # forms a cycle: 0→1→2→0
    (3, 2),           # node 3 only links to 2, and has NO incoming — should be low rank
    (4, 3), (4, 5),
    (5, 4), (5, 6),
    (6, 5),
    (7, 0),           # node 7 links to 0 but nothing links to 7 → low rank
    # node 3 has out_degree=1, but we'll create a true dangling node:
    # actually node 3 has out-edge to 2, so let's add a real dangling node
]

G_small = nx.DiGraph()
G_small.add_nodes_from(range(8))
G_small.add_edges_from(small_edges)
# remove outgoing edges from node 3 to make it a true dangling node
G_small.remove_edge(3, 2)

print(f"Nodes: {G_small.number_of_nodes()},  Edges: {G_small.number_of_edges()}")
print(f"Dangling nodes (out_degree=0): {[n for n in G_small.nodes if G_small.out_degree(n) == 0]}")
print(f"\nOut-degrees:  {dict(G_small.out_degree())}")
print(f"In-degrees:   {dict(G_small.in_degree())}")

## 3. Adjacency → Column-Stochastic Transition Matrix

**Convention: column = source node.**  
`M[i][j]` = probability of moving to page i when currently at page j.

Why column-stochastic? Because we multiply `r_new = M @ r_old` — each column distributes the current page's rank to its neighbors.

In [ ]:
def build_transition_matrix(G):
    """
    DiGraph → column-stochastic matrix M.
    Dangling columns are left as all-zeros here — fixed in the next step.
    Returns M, dangling_mask, ordered node list.
    """
    n = G.number_of_nodes()
    nodes = sorted(G.nodes())
    idx = {node: i for i, node in enumerate(nodes)}

    M = np.zeros((n, n))

    for j_node in nodes:
        j = idx[j_node]
        out_neighbors = list(G.successors(j_node))

        if out_neighbors:
            for i_node in out_neighbors:
                i = idx[i_node]
                M[i, j] += 1.0 / len(out_neighbors)
        # dangling node: column stays zero — handled separately

    dangling_mask = np.array([G.out_degree(nodes[j]) == 0 for j in range(n)])
    return M, dangling_mask, nodes


M_raw, dangling, node_list = build_transition_matrix(G_small)
n = len(node_list)

print("Raw transition matrix (dangling columns are all-zero):")
print(np.round(M_raw, 2))
print(f"\nColumn sums: {np.round(M_raw.sum(axis=0), 2)}")
print(f"\nDangling node indices: {np.where(dangling)[0]}  ← these columns sum to 0")

## 4. Fixing Dangling Nodes

A zero column breaks the stochastic property — rank leaks out of the system.  
Mathematically the Markov chain becomes absorbing (it gets trapped at the dangling node).

**Fix:** treat a dangling node as pointing to every page with probability 1/n.  
This models the real behavior: a user at a dead-end page will type a new URL.

After the fix, every column sums to exactly 1.

In [ ]:
def fix_dangling_nodes(M, dangling_mask):
    """Replace each all-zero column (dangling node) with uniform 1/n."""
    n = M.shape[0]
    M_fixed = M.copy()
    M_fixed[:, dangling_mask] = 1.0 / n   # broadcast sets entire column at once
    return M_fixed


M_stochastic = fix_dangling_nodes(M_raw, dangling)

print("After dangling fix:")
print(np.round(M_stochastic, 3))
print(f"\nColumn sums: {np.round(M_stochastic.sum(axis=0), 6)}")
print(f"All columns sum to 1: {np.allclose(M_stochastic.sum(axis=0), 1.0)}")

## 5. The Google Matrix and the Damping Factor

Even with all columns summing to 1, the graph can have:
- **Disconnected components** — rank can't flow between them, no unique stationary dist
- **Periodic cycles** — rank oscillates forever instead of converging

The damping factor d = 0.85 adds a uniform "teleportation" overlay:

$$G = d \cdot \tilde{M} + \frac{(1-d)}{n} \cdot \mathbf{1}\mathbf{1}^T$$

Every entry of G is now ≥ (1−d)/n > 0.  
A strictly positive column-stochastic matrix is **primitive** (irreducible + aperiodic).  
By **Perron-Frobenius**: it has a unique positive eigenvector for eigenvalue exactly 1.  
Every other eigenvalue satisfies |λᵢ| ≤ d < 1 — they all decay. That's the guarantee.

The constant d = 0.85 was chosen empirically by Brin and Page — it gives good rankings and convergence in ~50–100 iterations for web-scale graphs.

In [ ]:
def build_google_matrix(M_stochastic, d=0.85):
    """G = d * M + (1-d)/n * ones_matrix"""
    n = M_stochastic.shape[0]
    teleport = np.ones((n, n)) / n
    return d * M_stochastic + (1 - d) * teleport


G_matrix = build_google_matrix(M_stochastic, d=0.85)

print(f"Google matrix shape: {G_matrix.shape}")
print(f"Column sums (must be 1.0): {np.round(G_matrix.sum(axis=0), 8)}")
print(f"Min entry: {G_matrix.min():.6f}  ← strictly positive, Perron-Frobenius applies")

# show the eigenvalues to confirm structure
eigenvalues = np.linalg.eigvals(G_matrix.T)
eigenvalues_sorted = np.sort(np.abs(np.real(eigenvalues)))[::-1]
print(f"\nTop 3 eigenvalue magnitudes: {np.round(eigenvalues_sorted[:3], 6)}")
print(f"  λ₁ = 1.0 (dominant),  λ₂ ≤ d = 0.85 — spectral gap = {1 - eigenvalues_sorted[1]:.4f}")

## 6. Solver 1: Power Iteration

Apply G repeatedly until the rank vector stops changing.

$$r^{(t+1)} = G \cdot r^{(t)}$$

**Why it works:** expand r₀ in the eigenbasis of G. At step k:  

$$G^k r_0 = \sum_i c_i \lambda_i^k v_i$$

Since |λᵢ| ≤ d < 1 for i > 1, all components except the dominant eigenvector v₁ decay geometrically. Convergence rate ≈ d^k = 0.85^k per iteration.

In [ ]:
def power_iteration(G_matrix, tol=1e-10, max_iter=200):
    """
    Iterates r = G @ r until L1 change < tol.
    Returns (pagerank_vector, history_of_all_iterates).
    """
    n = G_matrix.shape[0]
    r = np.ones(n) / n          # start uniform — any positive vector works
    history = [r.copy()]

    for i in range(max_iter):
        r_new = G_matrix @ r
        delta = np.abs(r_new - r).sum()
        history.append(r_new.copy())
        r = r_new

        if delta < tol:
            print(f"Converged in {i+1} iterations  (delta = {delta:.2e})")
            break
    else:
        print("Warning: did not converge within max_iter")

    return r / r.sum(), np.array(history)  # normalize to be safe


pr_power, history = power_iteration(G_matrix)

print("\nPageRank (power iteration):")
for node, rank in zip(node_list, pr_power):
    bar = '█' * int(rank * 300)
    print(f"  Node {node}: {rank:.6f}  {bar}")

## 7. Solver 2: Eigendecomposition

PageRank is the **right eigenvector** of G for λ = 1:

$$G \mathbf{r} = \mathbf{r}$$

G is column-stochastic (columns sum to 1). The right eigenvector for λ=1 is the stationary distribution — PageRank.

`np.linalg.eig(G)` returns right eigenvectors directly. We pick the one whose eigenvalue is closest to 1 in magnitude (Perron-Frobenius says it's unique and positive).

**Common pitfall:** using `eig(G.T)` instead. Since G is column-stochastic, `1^T G = 1^T`, so the dominant left eigenvector of G (= right eigenvector of G^T) is just the all-ones vector — useless.

In [ ]:
def eigen_pagerank(G_matrix):
    """
    PageRank as the dominant RIGHT eigenvector of G.

    G is column-stochastic → G @ r = r → r is the right eigenvector for λ=1.
    np.linalg.eig(G) gives right eigenvectors directly.

    Do NOT use eig(G.T): since G is column-stochastic, 1^T G = 1^T, so the
    dominant right eigenvector of G^T is the all-ones vector — not PageRank.
    """
    eigenvalues, eigenvectors = np.linalg.eig(G_matrix)

    # dominant eigenvalue has largest magnitude (Perron-Frobenius: it equals 1)
    idx = np.argmax(np.abs(eigenvalues))
    dominant = np.real(eigenvectors[:, idx])  # imaginary part is float noise

    # Perron vector must be positive; flip sign if needed
    if dominant.sum() < 0:
        dominant = -dominant
    dominant = np.abs(dominant)   # clip any tiny negative float noise

    print(f"Dominant eigenvalue: {eigenvalues[idx]:.12f}  (theory: exactly 1.0)")
    return dominant / dominant.sum()


pr_eigen = eigen_pagerank(G_matrix)

print("\nPageRank (eigendecomposition):")
for node, rank in zip(node_list, pr_eigen):
    bar = '█' * int(rank * 300)
    print(f"  Node {node}: {rank:.6f}  {bar}")

## 8. Verification — Both Methods Must Agree

Any difference is floating-point noise, not a mathematical discrepancy. They're both computing the same eigenvector — one iteratively, one directly.

In [ ]:
diff = np.abs(pr_power - pr_eigen)

print(f"{'Node':<6} {'Power Iter':>12} {'Eigendecomp':>13} {'|Diff|':>12}")
print("-" * 46)
for node, p, e, d in zip(node_list, pr_power, pr_eigen, diff):
    print(f"{node:<6} {p:>12.8f} {e:>13.8f} {d:>12.2e}")

print(f"\nMax |diff|:  {diff.max():.2e}  ← pure floating-point noise")

# cross-check against NetworkX's built-in (uses the same algorithm)
pr_nx = nx.pagerank(G_small, alpha=0.85, tol=1e-12)
diff_nx = max(abs(pr_power[i] - pr_nx[node_list[i]]) for i in range(n))
print(f"Max diff vs networkx.pagerank: {diff_nx:.2e}")

rank_power = np.argsort(-pr_power)
rank_eigen = np.argsort(-pr_eigen)
print(f"Ranking order identical: {np.all(rank_power == rank_eigen)}")

## 9. Visualization

### 9a. Convergence of Power Iteration

Left: every node's score across iterations.  
Right: L1 distance from the final converged vector (log scale) — should track 0.85^t closely.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.tab10(np.linspace(0, 1, n))

# --- Left: per-node score evolution ---
ax = axes[0]
for i in range(n):
    ax.plot([h[i] for h in history], color=colors[i], lw=1.8, label=f"Node {i}")
ax.set_xlabel("Iteration")
ax.set_ylabel("PageRank score")
ax.set_title("Convergence of Each Node's Score")
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.grid(alpha=0.3)

# --- Right: L1 distance from final rank ---
ax2 = axes[1]
final = history[-1]
deltas = [np.abs(h - final).sum() for h in history[:-1]]

ax2.semilogy(deltas, 'crimson', lw=2, label='Actual L1 distance')

# theoretical bound: d^t scaled to match starting point
t = np.arange(len(deltas))
ax2.semilogy(t, deltas[0] * 0.85**t, 'k--', alpha=0.6, lw=1.5, label='0.85^t bound')

ax2.set_xlabel("Iteration")
ax2.set_ylabel("L1 distance from converged rank")
ax2.set_title("Convergence Rate (log scale)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("convergence.png", bbox_inches='tight')
plt.show()
print("The actual rate tracks the theoretical 0.85^t bound — spectral gap confirmed.")

### 9b. Graph — Node Size and Color Proportional to Rank

More important nodes are larger and darker.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

pos = nx.spring_layout(G_small, seed=7, k=2.5)

# scale node sizes: min 400, max 3000
r_min, r_max = pr_power.min(), pr_power.max()
sizes = 400 + (pr_power - r_min) / (r_max - r_min + 1e-12) * 2600

# color by rank using YlOrRd colormap
norm  = mcolors.Normalize(vmin=r_min, vmax=r_max)
node_colors = plt.cm.YlOrRd(norm(pr_power))

nx.draw_networkx_nodes(G_small, pos, node_size=sizes, node_color=node_colors, ax=ax, alpha=0.92)
nx.draw_networkx_labels(G_small, pos, ax=ax, font_size=10, font_weight='bold', font_color='black')
nx.draw_networkx_edges(
    G_small, pos, ax=ax,
    edge_color='#555555', arrows=True,
    arrowstyle='-|>', arrowsize=22,
    connectionstyle='arc3,rad=0.12', width=1.5, alpha=0.7
)

# colorbar
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='PageRank score', shrink=0.75, pad=0.02)

ax.set_title("PageRank — node size & color proportional to importance", fontsize=13, pad=12)
ax.axis('off')
plt.tight_layout()
plt.savefig("pagerank_graph.png", bbox_inches='tight')
plt.show()

## 10. Scaling Up — Stanford SNAP Dataset

Now run the exact same pipeline on a real-world graph from [Stanford SNAP](https://snap.stanford.edu/data/).  
We use `p2p-Gnutella08` — a peer-to-peer file-sharing network, ~6K nodes, ~20K directed edges.

If the download fails (no internet), a synthetic Barabási–Albert scale-free graph is used as a stand-in. Scale-free graphs have the same power-law structure as web link graphs.

In [ ]:
def load_snap_graph(url, local_path):
    """Download a SNAP .txt.gz edge list and return a DiGraph."""
    if not os.path.exists(local_path):
        print(f"Downloading {url} ...")
        urlretrieve(url, local_path)
        print("Download complete.")

    G = nx.DiGraph()
    with gzip.open(local_path, 'rt') as f:
        for line in f:
            if line.startswith('#'):  # SNAP files have comment headers
                continue
            parts = line.split()
            u, v = int(parts[0]), int(parts[1])
            if u != v:              # skip self-loops
                G.add_edge(u, v)
    return G


def make_fallback_graph(n=800, m=4):
    """Directed scale-free graph — same power-law structure as web graphs."""
    print("Using synthetic scale-free graph (fallback, no internet needed).")
    G_undir = nx.barabasi_albert_graph(n, m, seed=42)
    G = nx.DiGraph()
    rng = np.random.default_rng(42)
    for u, v in G_undir.edges():
        if rng.random() > 0.5:
            G.add_edge(u, v)
        else:
            G.add_edge(v, u)
    return G


try:
    SNAP_URL  = "https://snap.stanford.edu/data/p2p-Gnutella08.txt.gz"
    G_snap    = load_snap_graph(SNAP_URL, "p2p-Gnutella08.txt.gz")
    src_label = "SNAP p2p-Gnutella08"
except Exception as e:
    print(f"Download failed: {e}")
    G_snap    = make_fallback_graph(n=800, m=4)
    src_label = "Synthetic scale-free"

print(f"\n{src_label}: {G_snap.number_of_nodes()} nodes, {G_snap.number_of_edges()} edges")

In [ ]:
# Run the full pipeline — same three functions, bigger graph
t0 = time.time()

M_snap_raw, dangling_snap, nodes_snap = build_transition_matrix(G_snap)
M_snap_stoch  = fix_dangling_nodes(M_snap_raw, dangling_snap)
G_snap_matrix = build_google_matrix(M_snap_stoch, d=0.85)

pr_snap, history_snap = power_iteration(G_snap_matrix, tol=1e-8, max_iter=200)

elapsed = time.time() - t0
n_snap  = G_snap.number_of_nodes()

print(f"\nTime: {elapsed:.2f}s  ({n_snap} nodes, {n_snap**2 / 1e6:.1f}M matrix entries)")
print(f"Dangling nodes: {dangling_snap.sum()}  ({100*dangling_snap.mean():.1f}% of all nodes)")

# Top 10
top10 = np.argsort(-pr_snap)[:10]
print(f"\nTop 10 nodes by PageRank ({src_label}):")
print(f"  {'Rank':<5} {'Node':>8}  {'Score':>10}")
for pos, idx in enumerate(top10, 1):
    print(f"  #{pos:<4} {nodes_snap[idx]:>8}  {pr_snap[idx]:>10.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Convergence rate on SNAP graph ---
ax = axes[0]
final_snap = history_snap[-1]
deltas_snap = [np.abs(h - final_snap).sum() for h in history_snap[:-1]]
t = np.arange(len(deltas_snap))

ax.semilogy(deltas_snap, 'steelblue', lw=2, label='Actual')
ax.semilogy(t, deltas_snap[0] * 0.85**t, 'k--', alpha=0.6, label='0.85^t bound')
ax.set_xlabel("Iteration"); ax.set_ylabel("L1 distance")
ax.set_title(f"Convergence ({src_label})")
ax.legend(); ax.grid(alpha=0.3)

# --- PageRank score distribution (log-log) ---
ax2 = axes[1]
sorted_ranks = np.sort(pr_snap)[::-1]
ax2.loglog(np.arange(1, n_snap + 1), sorted_ranks, 'o', ms=2.5, color='steelblue', alpha=0.5)
ax2.set_xlabel("Rank (log)"); ax2.set_ylabel("Score (log)")
ax2.set_title("PageRank Distribution (log-log)\nPower-law tail")
ax2.grid(alpha=0.3)

# --- Out-degree vs PageRank (scatter) ---
ax3 = axes[2]
out_deg = np.array([G_snap.out_degree(nodes_snap[i]) for i in range(n_snap)])
ax3.scatter(out_deg + 1, pr_snap, alpha=0.15, s=4, color='tomato')
ax3.set_xscale('log'); ax3.set_yscale('log')
ax3.set_xlabel("Out-degree + 1 (log)"); ax3.set_ylabel("PageRank (log)")
ax3.set_title("Out-degree vs PageRank\nHigh out-degree ≠ high rank")
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("snap_analysis.png", bbox_inches='tight')
plt.show()

print("\nKey insight: out-degree barely correlates with PageRank.")
print("What matters is *who links to you*, not how many links you have outgoing.")

## 11. Why Power Iteration = Eigendecomposition (Proof Sketch)

Assume G is diagonalizable: $G = V \Lambda V^{-1}$, with eigenvalues $\lambda_1 \geq |\lambda_2| \geq \ldots$

Expand the initial vector in the eigenbasis:

$$r_0 = \sum_{i=1}^{n} c_i v_i$$

After k multiplications by G:

$$G^k r_0 = \sum_{i=1}^{n} c_i \lambda_i^k v_i = \lambda_1^k \left( c_1 v_1 + \sum_{i=2}^{n} c_i \left(\frac{\lambda_i}{\lambda_1}\right)^k v_i \right)$$

Since $\lambda_1 = 1$ and $|\lambda_i / \lambda_1| \leq d < 1$ for $i > 1$ (from Perron-Frobenius + damping factor), every term except $c_1 v_1$ **shrinks geometrically to zero**.

After normalizing: $G^k r_0 / \|G^k r_0\|_1 \to v_1$ as $k \to \infty$

That limit **is** the dominant eigenvector — exactly what eigendecomposition finds directly.

**Convergence rate:** The error at step k is $O\left((|\lambda_2|/\lambda_1)^k\right) = O(d^k)$.  
With d = 0.85, each iteration cuts the error by at least 15%.  
After 100 iterations: $0.85^{100} \approx 4 \times 10^{-8}$ — matches our convergence plot.

| Property | Without damping | With damping (d=0.85) |
|---|:---:|:---:|
| Unique stationary distribution | Not guaranteed | Yes |
| Convergence guaranteed | Not always | Yes |
| Spectral gap ≥ | 0 | 0.15 |
| All entries strictly positive | No | Yes |
| Perron-Frobenius applies | No | Yes |

## 12. Clean Reusable Pipeline

Everything above packed into a single `pagerank_from_scratch()` call.

In [ ]:
def pagerank_from_scratch(G, d=0.85, tol=1e-10, method='power'):
    """
    Full PageRank pipeline for any nx.DiGraph.

    Parameters
    ----------
    G      : nx.DiGraph
    d      : damping factor (0.85 standard)
    tol    : convergence tolerance for power iteration
    method : 'power' | 'eigen'

    Returns
    -------
    dict {node: rank_score}  — scores sum to 1
    """
    M, dangling_mask, node_list = build_transition_matrix(G)
    M_stoch = fix_dangling_nodes(M, dangling_mask)
    G_mat   = build_google_matrix(M_stoch, d=d)

    if method == 'power':
        ranks, _ = power_iteration(G_mat, tol=tol)
    else:
        ranks = eigen_pagerank(G_mat)

    return {node: float(r) for node, r in zip(node_list, ranks)}


# --- Final sanity check ---
# Compare power iteration vs eigendecomposition
pr_power_dict = pagerank_from_scratch(G_small, method='power')
pr_eigen_dict = pagerank_from_scratch(G_small, method='eigen')

print(f"\n{'Node':<6} {'Power':>10} {'Eigen':>10} {'|Diff|':>12}")
print("-" * 42)
for v in sorted(G_small.nodes()):
    p, e = pr_power_dict[v], pr_eigen_dict[v]
    print(f"{v:<6} {p:>10.7f} {e:>10.7f} {abs(p-e):>12.2e}")

max_diff = max(abs(pr_power_dict[v] - pr_eigen_dict[v]) for v in G_small.nodes())
print(f"\nMax |diff| power vs eigen: {max_diff:.2e}")
print("Match!" if max_diff < 1e-6 else f"Mismatch ({max_diff:.2e}) — check implementation")

# Check ranking order
nodes_sorted = sorted(G_small.nodes())
rank_power = sorted(nodes_sorted, key=lambda v: -pr_power_dict[v])
rank_eigen = sorted(nodes_sorted, key=lambda v: -pr_eigen_dict[v])
print(f"Identical ranking order: {rank_power == rank_eigen}")
print(f"Top-ranked node: {rank_power[0]}")